# Stage 1 — Entity-Resolution EDA (interactive companion)

This notebook **reuses the same package functions** as `scripts/run_eda.py` — it does not duplicate logic.
The canonical, reproducible run is always:

```bat
python scripts/run_eda.py --data-root dataset
```

Use this notebook to poke at intermediate tables/figures after (or before) that run.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent  # notebooks/ -> repo root
sys.path.insert(0, str(ROOT / 'src'))

from business_entity_resolution.config import load_config
from business_entity_resolution.io import load_test_tables, load_training_tables
from business_entity_resolution.validation import run_all_validations

cfg = load_config(project_root=ROOT)  # honors BER_DATA_ROOT / --data-root equivalent
print('project_root:', cfg.project_root)
print('data_root   :', cfg.data_root)

In [ ]:
train = load_training_tables(cfg)
test = load_test_tables(cfg)
print({k: (None if v is None else len(v)) for k, v in {**train, **test}.items()})
print(run_all_validations(cfg, train, test).to_console())

## 1. Dataset audit (same function as the pipeline)

In [ ]:
from business_entity_resolution.eda import run_dataset_audit

audit_res = run_dataset_audit({**train, **test}, cfg)
audit_res['n_rows']

## 2. Ground-truth match distribution

In [ ]:
from business_entity_resolution.eda import run_ground_truth_eda

gt_res = run_ground_truth_eda(train.get('train_gt'), cfg)
{k: v for k, v in gt_res.items() if k != 'artifacts'}

## 3. Normalization behavior on a few raw rows

Raw values are never overwritten — normalization only adds new columns.

In [ ]:
from business_entity_resolution.eda import ensure_normalized_columns

s1 = ensure_normalized_columns(train['train_s1'], cfg.columns)
s1[['business_name', 'name_norm', 'name_aggr', 'business_address', 'address_norm']].head(10)

## 4. Full reproducible run (everything: CSVs + figures + casebook + summary)

Prefer the script from a terminal so the run is recorded in `logs/run_history.jsonl` with the exact command.
The cell below runs the identical orchestrator for convenience (no run-history record from here).

In [ ]:
from business_entity_resolution.eda import run_full_eda

# NOTE: run_eda.py is the canonical entry point (it also validates + logs the run).
# out = run_full_eda(cfg, train, test, run_meta={'note': 'from-notebook'})
# print(f"{len(out['artifacts'])} artifacts")
print('Uncomment to run the full EDA from here, or run:  python scripts/run_eda.py --data-root dataset')